# Imports

In [20]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [39]:
#!pip install apify-client

import pandas as pd
import requests, re, time, random, os
from apify_client import ApifyClient
import json

In [40]:
folder_path = '' # Enter folder pathe here
spreadsheets = [file for file in os.listdir(f'{folder_path}/Raw Listings')]

# Function Definitions

In [41]:
USER_AGENTS = [] # List user agents here]
PROXIES = [] # List proxies here]

In [42]:
def get_airbnb_listing_html(url): #takes in url for listing and returns html
    headers = {
        "User-Agent": random.choice(USER_AGENTS)
    }
    proxy = {"http": random.choice(PROXIES), "https": random.choice(PROXIES)}

    try:
        r = requests.get(url, headers=headers, proxies=proxy, timeout=10)
        r.raise_for_status()
        return r.text
    except requests.exceptions.RequestException as e1:
        print(f"Request Error: {e1} | URL: {url}")
        return "RequestException"
    except IndexError as e2:
        print(f"Index Error: {e2} | URL: {url}")
        return "IndexError"


def get_airbnb_coordinates(html): #takes in html for listing and returns geographical coordinates
        if html is None:
            return None, None

        lat_pattern = re.compile(r'"lat":([-0-9.]+),')
        lng_pattern = re.compile(r'"lng":([-0-9.]+),')

        lat = lat_pattern.findall(html)[0]
        lng = lng_pattern.findall(html)[0]

        return lat, lng


def get_airbnb_bed_and_bath(html): #takes in html for listing and returns bedroom and bathroom count (studio = 0 bedrooms)
    if html is None:
        return None, None

    overviewItems_pattern = re.compile(r'"overviewItems":\[\s*(.*?)\s*\],"reviewData":')
    overviewItems = overviewItems_pattern.findall(html)

    bedrooms_pattern = re.compile(r'\b(\d+)\s*bedrooms?\b')
    bathrooms_pattern = re.compile(r'\b(\d+\.?\d?)\s*baths?\b')

    try:
        bedrooms = float(bedrooms_pattern.findall(overviewItems[0])[0])
    except IndexError as e:
        bedrooms = 0

    try:
        bathrooms = float(bathrooms_pattern.findall(overviewItems[0])[0])
    except IndexError as e:
        bathrooms = 1

    return bedrooms, bathrooms

def get_airbnb_superhost_status(html): #takes in html for listing and returns superhost status of the host
    if html is None:
        return None

    superHost_pattern = re.compile(r'"isSuperHost":"(true|false)"')
    isSuperHost = superHost_pattern.findall(html)[0]
    return str(isSuperHost)

def get_airbnb_amenities(html):
    if html is None:
        return None

    previewAmenities_pattern = re.compile(r'"previewAmenitiesGroups":\[\s*(.*?)\s*\],"seeAllAmenitiesGroups":')
    previewAmenities = previewAmenities_pattern.findall(html)

    amenities_pattern = re.compile(r'"title":"(.*?)"')
    amenities = amenities_pattern.findall(previewAmenities[0])

    return str(amenities)

In [53]:
def scrape(df_raw, df_listings, df_bookings, df_prices):

    for i in range(len(df_listings)):
        if df_listings.loc[i, "Scrape"] == False:
            continue

        listing_id = df_prices.loc[i, "ID"]
        url = f'https://www.airbnb.com/rooms/{listing_id}'
        html = get_airbnb_listing_html(url)

        if html == "RequestException":
            df_listings.loc[i, "Scrape"] = True

        else:
            df_listings.loc[i, "Scrape"] = False

            lat, lng = get_airbnb_coordinates(html)
            df_listings.loc[i, "Latitude"] = lat
            df_listings.loc[i, "Longitude"] = lng

            superhost_status = get_airbnb_superhost_status(html)
            df_listings.loc[i, "isSuperHost"] = superhost_status

            amenities = get_airbnb_amenities(html)
            df_listings.at[i, "Amenities"] = amenities


            if listing_id in list(df.ID):
                df_listings.loc[i, "Bedrooms"] = df[df.ID==listing_id]["Bedrooms"].values[0]
                df_listings.loc[i, "Bathrooms"] = df[df.ID==listing_id]["Bathrooms"].values[0]

            else:
                bedrooms, bathrooms = get_airbnb_bed_and_bath(html)
                df_listings.loc[i, "Bedrooms"] = bedrooms
                df_listings.loc[i, "Bathrooms"] = bathrooms

        if i % 5 == 0 or i == len(df_listings)-1:
            print(f'{i} records populated')
            df_bookings = df_bookings[['ID', 'January', 'February', 'March', 'April', 'May', 'June', 'July', 'August', 'September', 'October', 'November', 'December', 'Avg. Occupany %']]
            df_prices = df_prices[['ID', 'January', 'February', 'March', 'April', 'May', 'June', 'July', 'August', 'September', 'October', 'November', 'December']]
            df_bookings = pd.merge(df_bookings, df_listings, on = 'ID', how = 'left')
            df_prices = pd.merge(df_prices, df_listings, on = 'ID', how = 'left')

            df_bookings.to_csv(path_or_buf= f'{folder_path}/Enriched Listings/{os.path.splitext(spreadsheet)[0]}_bookings.csv', index=False)
            df_prices.to_csv(path_or_buf= f'{folder_path}/Enriched Listings/{os.path.splitext(spreadsheet)[0]}_prices.csv', index=False)

        # Random delay before next request
        time.sleep(random.uniform(2, 5))
    return df_bookings, df_prices

# Data Retrieval

### First Attempt

In [ ]:
for spreadsheet in spreadsheets:
    df = pd.read_excel(f'{folder_path}/Raw Listings/{spreadsheet}', sheet_name = 'General Listings Info')
    df_bookings = pd.read_excel(f'{folder_path}/Raw Listings/{spreadsheet}', sheet_name = 'Monthly Bookings')
    df_prices = pd.read_excel(f'{folder_path}/Raw Listings/{spreadsheet}', sheet_name = 'Monthly Avg. Nightly Price')

    df_listings = df_bookings[["ID"]].drop_duplicates().reset_index()
    df_listings["Scrape"] = True

    print(f'Started populating: {spreadsheet}')
    df_bookings_enriched, df_prices_enriched = scrape(df, df_listings, df_bookings, df_prices)
    print(f'Finished populating: {spreadsheet}')

### Second Attempt

In [ ]:
for spreadsheet in spreadsheets:
    df = pd.read_excel(f'{folder_path}/Raw Listings/{spreadsheet}', sheet_name = 'General Listings Info')
    df_bookings = pd.read_csv(f'{folder_path}/Enriched Listings/{os.path.splitext(spreadsheet)[0]}_bookings.csv')
    df_prices = pd.read_csv(f'{folder_path}/Enriched Listings/{os.path.splitext(spreadsheet)[0]}_prices.csv')

    df_listings = df_prices[["ID", "Scrape"]].drop_duplicates().reset_index()

    print(f'Started populating: {spreadsheet}')
    df_bookings_enriched, df_prices_enriched = scrape(df, df_listings, df_bookings, df_prices)
    print(f'Finished populating: {spreadsheet}')

### Combining Booking and Pricing Data

In [ ]:
for spreadsheet in spreadsheets:
    df = pd.read_excel(f'{folder_path}/Raw Listings/{spreadsheet}', sheet_name = 'General Listings Info')
    df_bookings_raw = pd.read_excel(f'{folder_path}/Raw Listings/{spreadsheet}', sheet_name = 'Monthly Bookings')
    df_prices_raw = pd.read_excel(f'{folder_path}/Raw Listings/{spreadsheet}', sheet_name = 'Monthly Avg. Nightly Price')

    df_bookings_raw = df_bookings_raw[['ID', 'January', 'February', 'March', 'April', 'May', 'June', 'July', 'August', 'September', 'October', 'November', 'December']]
    df_prices_raw = df_prices_raw[['ID', 'January', 'February', 'March', 'April', 'May', 'June', 'July', 'August', 'September', 'October', 'November', 'December']]

    # List of months to melt
    month_cols = ['January', 'February', 'March', 'April', 'May', 'June', 'July', 'August', 'September', 'October', 'November', 'December']

    def prepare_for_melt(df, value_name):
        # Create a 'Year_Index' to distinguish multiple rows for the same ID
        # (e.g., the 1st occurrence is Year 0, 2nd is Year 1, etc.)
        df = df.copy()
        df['Year_Index'] = df.groupby('ID').cumcount()

        # Melt from wide (months as columns) to long (months as rows)
        return df.melt(
            id_vars=['ID', 'Year_Index'],
            value_vars=month_cols,
            var_name='Month',
            value_name=value_name
        )

    # Transform both dataframes
    df_bookings_long = prepare_for_melt(df_bookings_raw, 'Bookings')
    df_prices_long = prepare_for_melt(df_prices_raw, 'Avg_Price')

    # Merge the two melted dataframes on ID, Month, and the generated Year_Index
    # This ensures the 1st occurrence of Jan in prices matches the 1st in bookings
    df_combined_long = pd.merge(
        df_bookings_long,
        df_prices_long,
        on=['ID', 'Year_Index', 'Month'],
        how='inner'
    )

    # Merge with the scraped info metadata
    df_scraped_info = pd.read_csv(f'{folder_path}/Enriched Listings/{os.path.splitext(spreadsheet)[0]}_bookings.csv')
    df_scraped_info = df_scraped_info[['ID', 'Avg. Occupany %', 'Scrape', 'Latitude', 'Longitude', 'isSuperHost', 'Amenities', 'Bedrooms', 'Bathrooms']].drop_duplicates().reset_index()


    # Final consolidated dataframe
    df_final = pd.merge(df_combined_long, df_scraped_info, on='ID', how='left')
    df_final.to_csv(path_or_buf= f'{folder_path}/Enriched Listings/{os.path.splitext(spreadsheet)[0]}_bookings_and_prices.csv', index=False)


    # Data Quality Checks
    print(spreadsheet)
    print(f"Rows before melt: {len(df_bookings_raw)}, {len(df_prices_raw)}")
    print(f"Rows after melt: {len(df_bookings_long)}, {len(df_prices_long)}, {len(df_combined_long)}")
    print(f"Rows after final merge: {len(df_final)}")